# SWANIRVAR v1.1.6 — Public Colab Launcher

**Run all. No Google Drive mount.**

This notebook asks you to upload the full-stack ZIP, builds everything under `/content`, starts the existing FastAPI backend and the original frontend server, and automatically prints/opens the temporary Colab-hosted URL.

Optional Gemini/BHASHINI/data API credentials can be supplied through Colab Secrets; no secrets are embedded in this notebook.


In [1]:

# CELL 1 — Upload the public full-stack ZIP
from google.colab import files
from pathlib import Path
import zipfile, shutil

print("=" * 78)
print("SWANIRVAR v1.1.6 — PUBLIC COLAB")
print("=" * 78)

candidates = sorted(Path("/content").glob("Swanirvar_v1_1_6_Public_Colab*.zip"))

if candidates:
    ZIP = candidates[0]
    print("✅ Found:", ZIP)
else:
    print("Upload: Swanirvar_v1_1_6_Public_Colab.zip")
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("No ZIP uploaded.")
    ZIP = Path("/content") / next(iter(uploaded))

if not zipfile.is_zipfile(ZIP):
    raise ValueError("Uploaded file is not a valid ZIP.")

print("Using:", ZIP)


SWANIRVAR v1.1.6 — PUBLIC COLAB
Upload: Swanirvar_v1_1_6_Public_Colab.zip


Saving Swanirvar_v1_1_6_Public_Colab.zip to Swanirvar_v1_1_6_Public_Colab.zip
Using: /content/Swanirvar_v1_1_6_Public_Colab.zip


In [2]:

# CELL 2 — Extract to local Colab storage only
from pathlib import Path
import zipfile, shutil

ROOT = Path("/content/swanirvar_public_runtime")

if ROOT.exists():
    shutil.rmtree(ROOT)

ROOT.mkdir(parents=True)

with zipfile.ZipFile(ZIP) as z:
    z.extractall(ROOT)

projects = list(ROOT.rglob("backend/app.py"))
if not projects:
    raise FileNotFoundError("backend/app.py was not found in the uploaded ZIP.")

PROJECT = projects[0].parent.parent
FRONTEND = PROJECT / "frontend"
BACKEND = PROJECT / "backend"

required = [
    FRONTEND / "index.html",
    FRONTEND / "package.json",
    FRONTEND / "vite.config.ts",
    FRONTEND / "server.ts",
    FRONTEND / "src" / "main.tsx",
    FRONTEND / "src" / "index.css",
    FRONTEND / "src" / "utils" / "domTranslator.ts",
    BACKEND / "app.py",
    PROJECT / "requirements.txt",
]

print("PROJECT :", PROJECT)
print("FRONTEND:", FRONTEND)
print("BACKEND :", BACKEND)

for p in required:
    print(("✅" if p.exists() else "❌"), p)

missing = [str(p) for p in required if not p.exists()]
if missing:
    raise RuntimeError("Package is incomplete:\n" + "\n".join(missing))

print("\n✅ Package validation passed")


PROJECT : /content/swanirvar_public_runtime/swanirvar_corrected_build
FRONTEND: /content/swanirvar_public_runtime/swanirvar_corrected_build/frontend
BACKEND : /content/swanirvar_public_runtime/swanirvar_corrected_build/backend
✅ /content/swanirvar_public_runtime/swanirvar_corrected_build/frontend/index.html
✅ /content/swanirvar_public_runtime/swanirvar_corrected_build/frontend/package.json
✅ /content/swanirvar_public_runtime/swanirvar_corrected_build/frontend/vite.config.ts
✅ /content/swanirvar_public_runtime/swanirvar_corrected_build/frontend/server.ts
✅ /content/swanirvar_public_runtime/swanirvar_corrected_build/frontend/src/main.tsx
✅ /content/swanirvar_public_runtime/swanirvar_corrected_build/frontend/src/index.css
✅ /content/swanirvar_public_runtime/swanirvar_corrected_build/frontend/src/utils/domTranslator.ts
✅ /content/swanirvar_public_runtime/swanirvar_corrected_build/backend/app.py
✅ /content/swanirvar_public_runtime/swanirvar_corrected_build/requirements.txt

✅ Package valida

In [3]:

# CELL 3 — Load optional Colab Secrets + install Python dependencies
from google.colab import userdata
import os, sys, subprocess

secret_names = [
    "GEMINI_API_KEY",
    "BHASHINI_INFERENCE_API_KEY",
    "BHASHINI_TOKEN",
    "BHASHINI_ASR_SERVICE_ID",
    "BHASHINI_TTS_SERVICE_ID",
    "BHASHINI_ASR_SERVICE_ID_BN",
    "BHASHINI_ASR_SERVICE_ID_HI",
    "BHASHINI_ASR_SERVICE_ID_TA",
    "BHASHINI_ASR_SERVICE_ID_TE",
    "BHASHINI_TTS_SERVICE_ID_BN",
    "BHASHINI_TTS_SERVICE_ID_HI",
    "BHASHINI_TTS_SERVICE_ID_TA",
    "BHASHINI_TTS_SERVICE_ID_TE",
    "DATA_GOV_IN_API_KEY",
    "AGMARKNET_API_KEY",
    "ENAM_API_KEY",
    "ORS_API_KEY",
]

loaded = []

for name in secret_names:
    try:
        value = userdata.get(name)
        if value:
            os.environ[name] = value
            loaded.append(name)
    except Exception:
        pass

print("Loaded Colab Secrets:", loaded if loaded else "none")

# Python backend dependencies
subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "-q",
        "-r", str(PROJECT / "requirements.txt")
    ],
    check=True
)

print("✅ Python dependencies installed")


Loaded Colab Secrets: none
✅ Python dependencies installed


In [4]:

# CELL 4 — Install frontend dependencies locally in /content
import subprocess, shutil, os

NODE_MODULES = FRONTEND / "node_modules"

if NODE_MODULES.exists():
    shutil.rmtree(NODE_MODULES)

env = os.environ.copy()
env["npm_config_include"] = "optional"

install = subprocess.run(
    [
        "npm", "install",
        "--legacy-peer-deps",
        "--include=optional",
        "--no-audit",
        "--no-fund",
        "--package-lock=false",
    ],
    cwd=str(FRONTEND),
    env=env,
    text=True,
    capture_output=True,
)

print(install.stdout[-12000:])

if install.returncode != 0:
    print("\n--- npm stderr ---")
    print(install.stderr[-16000:])
    raise RuntimeError("Frontend npm installation failed.")

print("✅ Frontend dependencies installed locally")



added 216 packages in 37s

✅ Frontend dependencies installed locally


In [5]:

# CELL 5 — Build the ORIGINAL frontend with Vite directly
# IMPORTANT: do NOT run "npm run build" here because that also
# invokes the package's optional server bundling step.

import subprocess, os

BIN = FRONTEND / "node_modules" / ".bin"
VITE = BIN / "vite"

if not VITE.exists():
    raise RuntimeError(f"Vite executable not found: {VITE}")

try:
    VITE.chmod(VITE.stat().st_mode | 0o111)
except Exception:
    pass

env = os.environ.copy()
env["PATH"] = str(BIN) + os.pathsep + env.get("PATH", "")

vite_version = subprocess.run(
    [str(VITE), "--version"],
    cwd=str(FRONTEND),
    env=env,
    text=True,
    capture_output=True,
)

print("Vite:", vite_version.stdout.strip())

if vite_version.returncode != 0:
    print(vite_version.stderr[-8000:])
    raise RuntimeError("Vite executable is not working.")

build = subprocess.run(
    [str(VITE), "build"],
    cwd=str(FRONTEND),
    env=env,
    text=True,
    capture_output=True,
)

print(build.stdout)

if build.returncode != 0:
    print("\n--- Vite build error ---")
    print(build.stderr[-16000:])
    raise RuntimeError("Frontend build failed.")

DIST = FRONTEND / "dist"

if not (DIST / "index.html").exists():
    raise RuntimeError("dist/index.html was not created.")

css_files = list(DIST.rglob("*.css"))

print("✅ Frontend build complete")
print("✅ dist/index.html")
print("✅ CSS files:", len(css_files))


Vite: vite/8.3.1 linux-x64 node-v20.19.0
vite v8.3.1 building client environment for production...
transforming...
✓ 1903 modules transformed.
rendering chunks...
computing gzip size...
dist/index.html                          1.73 kB │ gzip:   0.73 kB
dist/assets/index-BvBCe1_N.css         102.22 kB │ gzip:  17.19 kB
dist/assets/purify.es-Bvo9QlJ8.js       28.08 kB │ gzip:  11.08 kB
dist/assets/index.es-CePXdybr.js       151.37 kB │ gzip:  48.87 kB
dist/assets/html2canvas-ovOnjrNb.js    199.48 kB │ gzip:  46.77 kB
dist/assets/index-DtV5kCtc.js        1,487.48 kB │ gzip: 425.14 kB

✓ built in 4.52s

✅ Frontend build complete
✅ dist/index.html
✅ CSS files: 1


In [6]:

# CELL 6 — Start FastAPI + original frontend server and expose Colab URL
import subprocess, time, socket, requests, os
from google.colab import output

def port_open(port):
    try:
        with socket.create_connection(("127.0.0.1", port), timeout=1):
            return True
    except OSError:
        return False

# Stop stale listeners from an earlier run in this runtime.
subprocess.run(
    ["bash", "-lc",
     "fuser -k 8000/tcp 2>/dev/null || true; "
     "fuser -k 3000/tcp 2>/dev/null || true"],
    check=False
)

RUNTIME = PROJECT / "runtime"
RUNTIME.mkdir(exist_ok=True)

env = os.environ.copy()
env["NODE_ENV"] = "production"
env["SWANIRVAR_BACKEND_URL"] = "http://127.0.0.1:8000"
env["SWANIRVAR_UI_PORT"] = "3000"

# Backend
backend_log_path = RUNTIME / "backend.log"
backend_log = open(backend_log_path, "w", encoding="utf-8")

backend_process = subprocess.Popen(
    [
        sys.executable, "-m", "uvicorn",
        "backend.app:app",
        "--host", "0.0.0.0",
        "--port", "8000",
    ],
    cwd=str(PROJECT),
    env=env,
    stdout=backend_log,
    stderr=subprocess.STDOUT,
)

deadline = time.time() + 45
while time.time() < deadline:
    if port_open(8000):
        break
    if backend_process.poll() is not None:
        backend_log.flush()
        print(backend_log_path.read_text(errors="replace")[-12000:])
        raise RuntimeError("Backend exited during startup.")
    time.sleep(0.5)

if not port_open(8000):
    backend_log.flush()
    print(backend_log_path.read_text(errors="replace")[-12000:])
    raise RuntimeError("Backend did not start on port 8000.")

print("✅ FastAPI backend: :8000")

# Frontend: use the ORIGINAL frontend/server.ts.
TSX = FRONTEND / "node_modules" / ".bin" / "tsx"

if not TSX.exists():
    raise RuntimeError(
        "tsx executable not found in frontend/node_modules/.bin."
    )

try:
    TSX.chmod(TSX.stat().st_mode | 0o111)
except Exception:
    pass

frontend_log_path = RUNTIME / "frontend.log"
frontend_log = open(frontend_log_path, "w", encoding="utf-8")

frontend_process = subprocess.Popen(
    [str(TSX), "server.ts"],
    cwd=str(FRONTEND),
    env=env,
    stdout=frontend_log,
    stderr=subprocess.STDOUT,
)

deadline = time.time() + 45
while time.time() < deadline:
    if port_open(3000):
        break
    if frontend_process.poll() is not None:
        frontend_log.flush()
        print(frontend_log_path.read_text(errors="replace")[-16000:])
        raise RuntimeError("Frontend exited during startup.")
    time.sleep(0.5)

if not port_open(3000):
    frontend_log.flush()
    print(frontend_log_path.read_text(errors="replace")[-16000:])
    raise RuntimeError("Frontend did not start on port 3000.")

print("✅ Swanirvar frontend: :3000")

# Smoke tests
tests = [
    ("Frontend", "http://127.0.0.1:3000/"),
    ("Frontend API", "http://127.0.0.1:3000/api/health"),
    ("Backend", "http://127.0.0.1:8000/health"),
]

for label, url in tests:
    r = requests.get(url, timeout=15)
    print(f"\n{label}: HTTP {r.status_code}")
    print(r.text[:700])
    r.raise_for_status()

# Automatic Colab public/proxy URL for this runtime.
FRONTEND_URL = output.eval_js("google.colab.kernel.proxyPort(3000)")

print("\n" + "=" * 78)
print("🎉 SWANIRVAR HOSTED URL")
print("=" * 78)
print(FRONTEND_URL)

# Open automatically.
try:
    output.eval_js(f"window.open({FRONTEND_URL!r}, '_blank')")
except Exception:
    pass

SWANIRVAR_BACKEND_PROCESS = backend_process
SWANIRVAR_FRONTEND_PROCESS = frontend_process
SWANIRVAR_FRONTEND_URL = FRONTEND_URL


✅ FastAPI backend: :8000
✅ Swanirvar frontend: :3000

Frontend: HTTP 200
<!doctype html>
<html lang="en">
  <head>
    <meta charset="UTF-8" />
    <meta name="viewport" content="width=device-width, initial-scale=1.0" />
    <title>SWANIRVAR — Indians are the New Way of India</title>
    <meta name="description" content="SWANIRVAR — Sovereign national platform with 5-second brand motion identity preloader, interactive prototype map, national statistics, sovereign authentication portal, and vernacular voice intelligence." />
    <meta property="og:title" content="SWANIRVAR — Indians are the New Way of India" />
    <meta property="og:description" content="SWANIRVAR — Sovereign national platform with 5-second brand motion identity preloader, interactive prototype m

Frontend API: HTTP 200
{"status":"ok","time":"2026-09-25T13:35:40.327Z"}

Backend: HTTP 200
{"ok":true,"service":"swanirvar","version":"1.1.4"}

🎉 SWANIRVAR HOSTED URL
https://3000-m-s-kkb-usw1b1-fymtaejhv28e-b.us-west1-1.pro

## Runtime notes

- No Google Drive is mounted or required.
- The entire application runs under `/content` for this Colab runtime.
- The printed frontend URL is temporary and changes after the runtime ends.
- Optional AI/voice integrations require each user's own Colab Secrets.
- Never put API keys directly into the notebook or ZIP.
